# Repaso Clases 01 a 05 - Data Science I

| Seccion | Temas |
|---------|-------|
| Ejemplo 1 | Python y NumPy |
| Ejemplo 2 | Pandas - carga, filtrado, agrupacion |
| Ejemplo 3 | Estadistica descriptiva |
| Ejemplo 4 | Limpieza de datos, strings, NaN |
| Ejemplo 5 | Matplotlib + Seaborn |
| Guia de Graficos | Como elegir, colorear y formatear |


---
# Ejemplo 1: Python y NumPy

Repaso rapido de los bloques fundamentales del lenguaje que se usan en todo Data Science.

In [ ]:
# TIPOS DE DATOS
# list: ordenada, mutable. dict: pares clave-valor.
# Son la base de como Pandas y NumPy reciben los datos.

ventas_semana = [4500, 3200, 5100, 4800, 6000, 5500, 3900]
producto = {'nombre': 'Laptop', 'precio': 1299.99, 'stock': 15}

print('Lista:', ventas_semana)
print('Max:', max(ventas_semana), '| Min:', min(ventas_semana))
print('Dict:', producto)


In [ ]:
# FUNCIONES Y LIST COMPREHENSIONS
# Las funciones encapsulan logica reutilizable: pasos de preprocesamiento,
# transformaciones o reglas de negocio que se aplican a muchas columnas.

def clasificar_venta(valor):
    if valor >= 5000:   return 'Alta'
    elif valor >= 3500: return 'Media'
    else:               return 'Baja'

categorias = [clasificar_venta(v) for v in ventas_semana]
print('Ventas:   ', ventas_semana)
print('Categorias:', categorias)


## NumPy - Por que no usar listas?

NumPy almacena datos en **bloques de memoria fija** (igual que C): operaciones
**vectorizadas** aplicadas a todos los elementos sin loops visibles.
Para 1 millon de datos puede ser 100x mas rapido que listas de Python.

In [ ]:
import numpy as np

ventas = np.array([4500, 3200, 5100, 4800, 6000, 5500, 3900], dtype=float)

print('+ IVA 21%:', (ventas * 1.21).round(2))
print('Dias > 4500:', ventas[ventas > 4500])
print(f'Media: {ventas.mean():.0f} | Mediana: {np.median(ventas):.0f}')
print(f'Desvio std: {ventas.std():.0f} | Percentil 75: {np.percentile(ventas, 75):.0f}')


In [ ]:
# ARRAYS 2D - Matrices
# En ML los datos tienen forma de MATRIZ: filas=observaciones, cols=features.
# axis=1 -> resultado por fila | axis=0 -> resultado por columna

ventas_2d = np.array([
    [4500, 3200, 5100, 4800],  # Region Norte
    [2800, 4100, 3600, 5200],  # Region Sur
    [6000, 5500, 4800, 7100],  # Region Centro
])
print('Shape:', ventas_2d.shape)
print('Total por region   (axis=1):', ventas_2d.sum(axis=1))
print('Promedio por producto (axis=0):', ventas_2d.mean(axis=0).round(0))


---
# Ejemplo 2: Pandas - Carga y Analisis

- **vs Excel**: maneja millones de filas, automatiza procesos, se integra con ML
- **vs SQL**: en memoria, sintaxis Python, facil de combinar con graficos
- **vs NumPy**: nombres de columnas, indices, NaN nativo, groupby, merge

> **ventas.csv** — columnas: `Region`, `Producto`, `Mes`, `Ventas`, `Satisfaccion_Cliente`, `Costo_Operativo`

In [ ]:
import pandas as pd
import numpy as np

# COMO LEER UN CSV
# Opcion A - archivo en la misma carpeta:
#   df = pd.read_csv('ventas.csv', encoding='utf-8')
#
# Opcion B - subir el archivo directamente en Colab (sin Drive):
#   from google.colab import files
#   uploaded = files.upload()
#   df = pd.read_csv('ventas.csv', encoding='utf-8')
#
# Opcion C - montar Google Drive:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   import os; os.chdir('/content/drive/MyDrive/Data Science 1/Repaso 01 a 05')
#   df = pd.read_csv('ventas.csv', encoding='utf-8')

df = pd.read_csv('ventas.csv', encoding='utf-8')

print('Shape:', df.shape)
print(df.head())
print(df.dtypes)


In [ ]:
# describe(): primer vistazo a distribuciones
# count < total filas -> hay NaN | mean muy distinta de mediana -> sesgo o outliers
print(df.describe().round(2))
print()
print('Regiones:', df['Region'].unique())
print('Productos:', df['Producto'].unique())


In [ ]:
# Filtrado booleano con & (AND) y | (OR) — cada condicion entre parentesis
norte_alta = df[(df['Region'] == 'Norte') & (df['Ventas'] > 4000)]
print(f'Norte con Ventas > 4000: {len(norte_alta)} filas')
print(norte_alta.head())


## GroupBy - Split, Apply, Combine

Divide el DataFrame en grupos, aplica una funcion a cada uno y combina resultados.
Equivalente al `GROUP BY` de SQL y las tablas dinamicas de Excel.

In [ ]:
resumen_region = df.groupby('Region').agg(
    ventas_total=('Ventas', 'sum'),
    ventas_promedio=('Ventas', 'mean'),
    satisfaccion_media=('Satisfaccion_Cliente', 'mean'),
    costo_total=('Costo_Operativo', 'sum'),
).round(2)
resumen_region['margen'] = resumen_region['ventas_total'] - resumen_region['costo_total']
print(resumen_region)


In [ ]:
tabla = pd.pivot_table(df, values='Ventas', index='Region',
                       columns='Producto', aggfunc='mean').round(0)
print('Ventas promedio (Region x Producto):')
print(tabla)

df['Margen'] = df['Ventas'] - df['Costo_Operativo']
df['Margen_Pct'] = (df['Margen'] / df['Ventas'] * 100).round(2)
print(df[['Region','Producto','Ventas','Margen','Margen_Pct']].head())


---
# Ejemplo 3: Estadistica Descriptiva

La estadistica descriptiva resume y describe los datos **sin hacer inferencias** sobre
una poblacion mayor. Es el primer paso obligatorio antes de cualquier modelo.

Se divide en dos grandes grupos:
- **Medidas de tendencia central**: donde se concentran los datos (media, mediana, moda)
- **Medidas de dispersion**: que tan separados estan los datos (varianza, desvio, IQR)

## Medidas de tendencia central

Las tres medidas describen el 'centro' de los datos, pero de formas distintas:

| Medida | Definicion | Cuando usarla |
|--------|-----------|---------------|
| **Media** | Suma de valores / cantidad | Distribucion simetrica, sin outliers |
| **Mediana** | Valor del medio al ordenar los datos | Hay outliers o la distribucion es asimetrica |
| **Moda** | Valor que mas se repite | Variables categoricas o discretas |

> Si la media y la mediana son muy distintas, hay **sesgo** o **outliers** en los datos.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('ventas.csv', encoding='utf-8')

# ============================================================
# MEDIDAS DE TENDENCIA CENTRAL
# ============================================================
ventas = df['Ventas']

media   = ventas.mean()
mediana = ventas.median()
moda    = ventas.mode()[0]  # mode() devuelve una Serie, tomamos el primer valor

print('--- Ventas ---')
print(f'Media:   {media:.0f}   <- suma de todos / cantidad')
print(f'Mediana: {mediana:.0f}  <- valor del medio al ordenar')
print(f'Moda:    {moda:.0f}  <- valor que mas se repite')
print()
dif = abs(media - mediana)
print(f'Diferencia media-mediana: {dif:.0f}')
if dif > media * 0.05:
    print('-> Diferencia notable: puede haber outliers o sesgo')
else:
    print('-> Distribucion aproximadamente simetrica')


## Medidas de dispersion

Describen **que tan separados** estan los datos respecto al centro.
Dos datasets pueden tener la misma media pero distribuciones completamente distintas.

| Medida | Formula / concepto | Cuando usarla |
|--------|--------------------|---------------|
| **Rango** | max - min | Rapido pero muy sensible a outliers |
| **Varianza** | promedio de las diferencias al cuadrado | Base del desvio estandar |
| **Desvio estandar** | raiz cuadrada de la varianza | Misma unidad que los datos, mas interpretable |
| **IQR** | Q3 - Q1 (rango intercuartil) | Robusto: no se afecta por outliers |

> Regla practica: usa **desvio estandar** si la distribucion es normal,
> usa **IQR** si hay outliers.

In [ ]:
# ============================================================
# MEDIDAS DE DISPERSION
# ============================================================
rango    = ventas.max() - ventas.min()
varianza = ventas.var()   # promedio de (xi - media)^2
desvio   = ventas.std()   # raiz(varianza), en la misma unidad que los datos

Q1  = ventas.quantile(0.25)  # el 25% de los datos esta por debajo
Q3  = ventas.quantile(0.75)  # el 75% de los datos esta por debajo
IQR = Q3 - Q1               # rango del 50% central

print('--- Dispersion de Ventas ---')
print(f'Rango:    {rango:.0f}  <- max - min (sensible a outliers)')
print(f'Varianza: {varianza:.0f}  <- en unidades al cuadrado, poco intuitiva')
print(f'Desvio:   {desvio:.0f}  <- misma unidad que los datos')
print(f'Q1:       {Q1:.0f}  <- el 25% de ventas esta por debajo')
print(f'Q3:       {Q3:.0f}  <- el 75% de ventas esta por debajo')
print(f'IQR:      {IQR:.0f}  <- rango del 50% central (robusto)')


## Percentiles

El **percentil N** indica que el N% de los datos esta por debajo de ese valor.

Ejemplos:
- Percentil 50 = mediana
- Percentil 25 = Q1 (primer cuartil)
- Percentil 75 = Q3 (tercer cuartil)
- Percentil 90 = el 10% de los datos mas altos empieza aca

Uso tipico en DS:
- Detectar outliers: valores por encima del percentil 99 suelen ser errores
- Segmentar clientes: 'top 10%' por gasto
- Normalizar por rango: `MinMaxScaler` usa min (p0) y max (p100)

In [ ]:
# ============================================================
# PERCENTILES
# ============================================================
percentiles = [10, 25, 50, 75, 90, 95, 99]

print('Percentiles de Ventas:')
for p in percentiles:
    val = ventas.quantile(p/100)
    print(f'  P{p:>2}: {val:>8.0f}  -> el {p}% de las ventas esta por debajo')

# Uso practico: filtrar outliers por encima del percentil 99
p99 = ventas.quantile(0.99)
outliers_p99 = df[df['Ventas'] > p99]
print(f'\nVentas por encima del P99 ({p99:.0f}): {len(outliers_p99)} registros')


## Deteccion de outliers con IQR

El metodo IQR es el mas robusto para detectar outliers porque no usa la media
(que ya esta contaminada por los valores extremos).

**Limites:**
- Limite inferior = Q1 - 1.5 × IQR
- Limite superior = Q3 + 1.5 × IQR

Cualquier valor fuera de esos limites se considera **outlier candidato**.
El paso siguiente es investigar: puede ser un error de carga o un caso real excepcional.

In [ ]:
# ============================================================
# DETECCION DE OUTLIERS CON IQR
# ============================================================
import matplotlib.pyplot as plt

lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers = df[(df['Ventas'] < lim_inf) | (df['Ventas'] > lim_sup)]

print(f'Q1={Q1:.0f}  Q3={Q3:.0f}  IQR={IQR:.0f}')
print(f'Limite inferior: {lim_inf:.0f}')
print(f'Limite superior: {lim_sup:.0f}')
print(f'Outliers detectados: {len(outliers)} de {len(df)} registros')

# Visualizar con boxplot: los puntos fuera de los bigotes son outliers
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot manual con Matplotlib
axes[0].boxplot(ventas.dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[0].axhline(lim_sup, color='red', linestyle='--', label=f'Lim sup: {lim_sup:.0f}')
axes[0].axhline(lim_inf, color='orange', linestyle='--', label=f'Lim inf: {lim_inf:.0f}')
axes[0].set_title('Boxplot de Ventas con limites IQR')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Histograma con los limites marcados
axes[1].hist(ventas.dropna(), bins=25, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(lim_sup, color='red', linestyle='--', linewidth=2, label=f'Lim sup: {lim_sup:.0f}')
axes[1].axvline(Q1, color='green', linestyle=':', linewidth=2, label=f'Q1: {Q1:.0f}')
axes[1].axvline(ventas.median(), color='black', linestyle='-', linewidth=2, label=f'Mediana: {ventas.median():.0f}')
axes[1].axvline(Q3, color='green', linestyle=':', linewidth=2, label=f'Q3: {Q3:.0f}')
axes[1].set_title('Histograma con mediana y cuartiles')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


## Estadistica por grupos

En DS casi nunca analizamos la estadistica global: la analizamos **por grupo**.
La media global puede ocultar diferencias importantes entre regiones o categorias.

In [ ]:
# ============================================================
# ESTADISTICA DESCRIPTIVA POR GRUPO
# ============================================================
import matplotlib.pyplot as plt

# describe() por grupo: equivale a un resumen completo por categoria
print('Estadisticas de Ventas por Region:')
print(df.groupby('Region')['Ventas'].describe().round(1))
print()

# Coeficiente de variacion: desvio / media * 100
# Mide la dispersion RELATIVA (independiente de la escala)
# CV alto = datos muy dispersos respecto al promedio
cv = df.groupby('Region')['Ventas'].agg(
    media='mean', desvio='std'
)
cv['coef_variacion_%'] = (cv['desvio'] / cv['media'] * 100).round(1)
print('Coeficiente de variacion por Region:')
print(cv)
print()
print('Interpretacion: CV alto -> ventas muy irregulares entre meses')


In [ ]:
# ============================================================
# CORRELACION DE PEARSON
# ============================================================
# La correlacion mide la relacion LINEAL entre dos variables numericas.
# Valor entre -1 y +1:
#   +1 = cuando una sube la otra sube exactamente igual
#   -1 = cuando una sube la otra baja exactamente igual
#    0 = no hay relacion lineal (puede haber relacion no lineal)

df['Margen'] = df['Ventas'] - df['Costo_Operativo']
cols = ['Ventas', 'Satisfaccion_Cliente', 'Costo_Operativo', 'Margen']
corr = df[cols].corr().round(3)

print('Matriz de correlacion de Pearson:')
print(corr)
print()
print('Pares mas correlacionados:')
# Extraer pares unicos (triangulo superior sin la diagonal)
import itertools
pares = [(c1, c2, corr.loc[c1,c2]) for c1,c2 in itertools.combinations(cols,2)]
for c1, c2, val in sorted(pares, key=lambda x: abs(x[2]), reverse=True):
    direccion = 'positiva' if val > 0 else 'negativa'
    print(f'  {c1} vs {c2}: {val:+.3f} ({direccion})')


---
# Ejemplo 4: Limpieza de Datos - Strings y NaN

**GIGO** (Garbage In, Garbage Out): un modelo entrenado con datos sucios aprende los errores.

Problemas mas comunes: strings sucios, NaN, duplicados.

In [ ]:
import pandas as pd
import numpy as np

df_original = pd.read_csv('ventas.csv', encoding='utf-8')
df = df_original.copy()

df.loc[[0,5,10], 'Region'] = '  norte '
df.loc[[1,6,11], 'Region'] = 'NORTE'
df.loc[[2,7,12], 'Region'] = 'sur '

np.random.seed(42)
df.loc[np.random.choice(len(df), int(len(df)*0.10), replace=False), 'Ventas'] = np.nan
df.loc[np.random.choice(len(df), int(len(df)*0.08), replace=False), 'Satisfaccion_Cliente'] = np.nan
df = pd.concat([df, df.iloc[:3]], ignore_index=True)

print('NaN:', df.isnull().sum().to_dict())
print('Duplicados:', df.duplicated().sum())
print('Regiones (con errores):', df['Region'].unique())


## `.str` — Limpiar strings sin loops

`'Norte'`, `'NORTE'`, `'  norte '` son **tres valores distintos** para Python.
Un `groupby` los trataria como tres grupos separados.
El patron `.str.strip().str.title()` normaliza todo en una linea.

In [ ]:
df_clean = df.copy()
df_clean['Region']   = df_clean['Region'].str.strip().str.title()
df_clean['Producto'] = df_clean['Producto'].str.strip().str.title()

print('ANTES:', df['Region'].unique())
print('DESPUES:', df_clean['Region'].unique())

df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f'Filas: {len(df)} -> {len(df_clean)} (sin duplicados)')


## Tratamiento de NaN

| Estrategia | Metodo | Cuando usarla |
|------------|--------|---------------|
| Eliminar | `dropna()` | Pocos NaN y aleatorios |
| Media | `fillna(mean)` | Distribucion simetrica, sin outliers |
| Mediana | `fillna(median)` | Hay valores extremos |
| Forward fill | `ffill()` | Series de tiempo: ultimo valor conocido |
| Backward fill | `bfill()` | Series de tiempo: proximo valor conocido |

In [ ]:
df_clean['Ventas'] = df_clean['Ventas'].fillna(df_clean['Ventas'].median())
df_clean['Satisfaccion_Cliente'] = df_clean['Satisfaccion_Cliente'].fillna(
    df_clean['Satisfaccion_Cliente'].mean())

print('NaN restantes:', df_clean.isnull().sum().sum())


In [ ]:
# ffill y bfill: para series de tiempo
temps = pd.Series([22.0, 23.5, None, None, 24.1, None, 25.0],
                  index=pd.date_range('2023-01-01', periods=7, freq='h'))

print('Original: ', temps.values)
print('ffill:    ', temps.ffill().values)
print('bfill:    ', temps.bfill().values)


---
# Ejemplo 5: Visualizaciones con Matplotlib y Seaborn

**Matplotlib**: control total, verbosa, ideal para el grafico final de presentacion.
**Seaborn**: abstraccion sobre Matplotlib, graficos estadisticos con una linea, ideal para EDA.

> Regla practica: explora con Seaborn, presentas con Matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

df = pd.read_csv('ventas.csv', encoding='utf-8')
df['Margen'] = df['Ventas'] - df['Costo_Operativo']
df['mes_num'] = df['Mes'].str.split('-').str[1].astype(int)
ts = df.groupby(['Region', 'mes_num'])['Ventas'].sum().reset_index()
print('Dataset listo:', df.shape)


## Como personalizar un grafico en Matplotlib

| Metodo en `ax` | Que controla |
|----------------|--------------|
| `set_title('t', fontsize=14, fontweight='bold')` | Titulo |
| `set_xlabel` / `set_ylabel` | Etiquetas de ejes |
| `set_xlim(a,b)` / `set_ylim(a,b)` | Rango visible |
| `set_xticks` + `set_xticklabels` | Marcas y nombres eje X |
| `legend(title='', loc='upper right')` | Leyenda |
| `grid(True, linestyle='--', alpha=0.4)` | Grilla |
| `text(x, y, 'texto', ha='center')` | Texto en coordenadas |
| `spines['top'].set_visible(False)` | Ocultar bordes |
| `figsize=(ancho, alto)` en `subplots` | Tamano en pulgadas |

In [ ]:
# GRAFICO 1: LINEPLOT TEMPORAL
meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
estilos = {'Norte': ('steelblue','o'), 'Sur': ('tomato','s'), 'Centro': ('seagreen','^')}

fig, ax = plt.subplots(figsize=(12, 5))
for region, (color, marker) in estilos.items():
    d = ts[ts['Region']==region].sort_values('mes_num')
    ax.plot(d['mes_num'], d['Ventas']/1000, color=color, marker=marker,
            linewidth=2, markersize=7, label=region)

ax.set_title('Ventas Mensuales por Region', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Mes'); ax.set_ylabel('Ventas (miles $)')
ax.set_xticks(range(1,13)); ax.set_xticklabels(meses, rotation=45, ha='right')
ax.legend(title='Region'); ax.grid(True, linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# GRAFICO 2: BARRAS AGRUPADAS CON ETIQUETAS
resumen = df.groupby(['Region','Producto'])['Ventas'].mean().unstack()
x = np.arange(len(resumen))
palette = ['#4C72B0','#DD8452','#55A868','#C44E52']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (prod, color) in enumerate(zip(resumen.columns, palette)):
    offset = (i - len(resumen.columns)/2 + 0.5) * 0.2
    bars = ax.bar(x+offset, resumen[prod]/1000, width=0.2, label=prod,
                  color=color, edgecolor='white')
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
                f'{b.get_height():.1f}K', ha='center', va='bottom', fontsize=8)

ax.set_title('Ventas Promedio por Region y Producto', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(resumen.index)
ax.set_ylim(0, resumen.values.max()/1000*1.25)
ax.legend(title='Producto'); ax.grid(True, axis='y', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# GRAFICO 3: HISTPLOT + KDE
# kde=True agrega la curva de densidad suavizada: muestra la forma de la
# distribucion sin depender del numero de bins elegido.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x='Ventas', hue='Region', kde=True, bins=20, alpha=0.45, ax=axes[0])
axes[0].set_title('Distribucion de Ventas por Region')
axes[0].grid(True, linestyle='--', alpha=0.3)

sns.histplot(data=df, x='Satisfaccion_Cliente', hue='Region', kde=True, bins=15, alpha=0.45, ax=axes[1])
axes[1].set_title('Distribucion de Satisfaccion por Region')
axes[1].grid(True, linestyle='--', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# GRAFICO 4: BOXPLOT + VIOLINPLOT
# Boxplot: caja=Q1 a Q3, linea=mediana, bigotes=1.5*IQR, puntos=outliers
# Violinplot: boxplot + forma completa de la distribucion (KDE rotado)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df, x='Producto', y='Ventas', hue='Region', palette='Set2', ax=axes[0])
axes[0].set_title('Boxplot: Ventas por Producto y Region')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, axis='y', linestyle='--', alpha=0.4)

sns.violinplot(data=df, x='Region', y='Ventas', palette='Set3', inner='quartile', ax=axes[1])
axes[1].set_title('Violinplot: Distribucion por Region')
axes[1].grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout(); plt.show()


In [ ]:
# GRAFICO 5: HEATMAP DE CORRELACIONES
# cmap='coolwarm': azul=negativo, blanco=cero, rojo=positivo

corr = df[['Ventas','Satisfaccion_Cliente','Costo_Operativo','Margen']].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlacion entre Variables Numericas', pad=12)
plt.tight_layout(); plt.show()


In [ ]:
# GRAFICO 6: SCATTER + REGRESION
# regplot: scatter + linea de regresion + banda de confianza al 95%
# banda angosta = relacion lineal solida

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df, x='Costo_Operativo', y='Ventas',
                hue='Region', style='Producto', alpha=0.7, s=60, ax=axes[0])
axes[0].set_title('Ventas vs Costo Operativo')
axes[0].grid(True, linestyle='--', alpha=0.3)

sns.regplot(data=df, x='Satisfaccion_Cliente', y='Ventas',
            scatter_kws={'alpha':0.5,'s':30,'color':'steelblue'},
            line_kws={'color':'red','linewidth':2}, ci=95, ax=axes[1])
axes[1].set_title('Ventas vs Satisfaccion (regresion + IC 95%)')
axes[1].grid(True, linestyle='--', alpha=0.3)
plt.tight_layout(); plt.show()


---
# Guia de Graficos: Como elegir, colorear y formatear

## 1. Que grafico usar segun el dato

| Situacion | Grafico recomendado | Funcion |
|-----------|--------------------|---------|
| Distribucion de una variable numerica | Histograma + KDE | `sns.histplot(kde=True)` |
| Comparar distribucion entre grupos | Boxplot o Violinplot | `sns.boxplot` / `sns.violinplot` |
| Evolucion en el tiempo | Lineplot | `ax.plot` / `sns.lineplot` |
| Comparar categorias | Barras | `ax.bar` / `sns.barplot` |
| Relacion entre dos variables numericas | Scatter | `sns.scatterplot` / `sns.regplot` |
| Relacion entre TODAS las variables | Matriz de correlacion | `sns.heatmap(df.corr())` |
| Proporcion de un total | Pie chart | `ax.pie` |
| Distribucion por pares de variables | Pairplot | `sns.pairplot` |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

df = pd.read_csv('ventas.csv', encoding='utf-8')
df['Margen'] = df['Ventas'] - df['Costo_Operativo']
print('CSV cargado:', df.shape)


## 2. Colores en Matplotlib

- **Nombre**: `'steelblue'`, `'tomato'`, `'seagreen'`, `'gold'`
- **Hex**: `'#4C72B0'` — igual que HTML/CSS
- **RGB normalizado**: `(0.3, 0.5, 0.8)` — valores entre 0 y 1
- **Letra corta**: `'b'`=blue, `'r'`=red, `'g'`=green, `'k'`=black

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
fig.suptitle('Formas de especificar colores en Matplotlib', fontsize=12, fontweight='bold')

axes[0].bar(['A','B','C'], [3,5,4], color=['steelblue','tomato','seagreen'])
axes[0].set_title('Por nombre')

axes[1].bar(['A','B','C'], [3,5,4], color=['#E63946','#457B9D','#2A9D8F'])
axes[1].set_title('Por hex (#RRGGBB)')

axes[2].bar(['A','B','C'], [3,5,4], color=[(0.9,0.3,0.1),(0.1,0.5,0.8),(0.2,0.7,0.3)])
axes[2].set_title('Por RGB (0 a 1)')

axes[3].bar(['A','B','C'], [3,5,4], color=['b','r','g'])
axes[3].set_title('Por letra corta')

for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


## 3. Paletas de colores en Seaborn (`palette=`)

| Paleta | Tipo | Mejor para |
|--------|------|------------|
| `'Set1'`, `'Set2'`, `'Set3'` | Categorica | Variables sin orden (regiones, categorias) |
| `'tab10'`, `'tab20'` | Categorica | Muchos grupos distintos |
| `'Blues'`, `'Greens'`, `'Oranges'` | Secuencial | Variable de bajo a alto |
| `'coolwarm'`, `'RdBu'` | Divergente | Valores positivos y negativos |
| `'viridis'`, `'magma'`, `'plasma'` | Perceptual | Recomendadas para impresion en grises |

In [ ]:
paletas = ['Set1', 'Set2', 'tab10', 'Blues', 'coolwarm', 'viridis']
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
fig.suptitle('Paletas de colores en Seaborn (palette=)', fontsize=13, fontweight='bold')

for ax, pal in zip(axes.flatten(), paletas):
    sns.barplot(data=df, x='Region', y='Ventas', palette=pal, ax=ax, errorbar=None)
    ax.set_title(f"palette='{pal}'", fontsize=10)
    ax.set_xlabel(''); ax.tick_params(axis='x', rotation=15)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout(); plt.show()


## 4. Estilos globales (`plt.style.use`)

| Estilo | Look |
|--------|------|
| `'default'` | Clasico de Matplotlib |
| `'seaborn-v0_8'` | Fondo gris claro, grilla blanca |
| `'ggplot'` | Inspirado en R ggplot2 |
| `'fivethirtyeight'` | Estilo del blog FiveThirtyEight |
| `'dark_background'` | Fondo negro |

In [ ]:
estilos_demo = ['default', 'seaborn-v0_8', 'ggplot', 'fivethirtyeight']
totales = df.groupby('Region')['Ventas'].sum() / 1000

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('El mismo grafico con distintos estilos', fontsize=13, fontweight='bold')

for ax, estilo in zip(axes, estilos_demo):
    with plt.style.context(estilo):
        ax.bar(totales.index, totales.values, color='steelblue')
        ax.set_title(f"'{estilo}'", fontsize=9)
        ax.set_ylabel('Ventas (miles $)')

plt.tight_layout(); plt.show()


## 5. Tipos de lineas y marcadores

**`linestyle=`** — `'-'` (solida), `'--'` (guiones), `':'` (puntos), `'-.'` (punto-guion)

**`marker=`** — `'o'` (circulo), `'s'` (cuadrado), `'^'` (triangulo), `'D'` (diamante), `'*'` (estrella)

In [ ]:
x = np.linspace(0, 10, 15)
configs = [
    ('-',  'o', 'steelblue', 'solida + circulo'),
    ('--', 's', 'tomato',    'guion + cuadrado'),
    (':',  '^', 'seagreen',  'puntos + triangulo'),
    ('-.', 'D', 'purple',    'punto-guion + diamante'),
]

fig, ax = plt.subplots(figsize=(10, 4))
for i, (ls, mk, color, label) in enumerate(configs):
    ax.plot(x, np.sin(x)+i*0.5, linestyle=ls, marker=mk, color=color,
            linewidth=2, markersize=8, label=label)

ax.set_title('Estilos de linea y marcador', fontsize=13)
ax.legend(); ax.grid(True, linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


## 6. El mismo dato con distintos tipos de grafico Seaborn

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('El mismo dato con distintos graficos', fontsize=13, fontweight='bold')

sns.barplot(data=df, x='Region', y='Ventas', palette='Set2',
            errorbar='sd', ax=axes[0,0])
axes[0,0].set_title('barplot (promedio + desvio std)')

sns.boxplot(data=df, x='Region', y='Ventas', palette='Set2', ax=axes[0,1])
axes[0,1].set_title('boxplot (mediana, cuartiles, outliers)')

sns.violinplot(data=df, x='Region', y='Ventas', palette='Set2',
               inner='quartile', ax=axes[0,2])
axes[0,2].set_title('violinplot (forma completa)')

sns.stripplot(data=df, x='Region', y='Ventas', palette='Set2',
              alpha=0.5, jitter=True, ax=axes[1,0])
axes[1,0].set_title('stripplot (cada punto visible)')

sns.swarmplot(data=df, x='Region', y='Ventas', palette='Set2',
              size=4, ax=axes[1,1])
axes[1,1].set_title('swarmplot (sin solaparse)')

sns.boxplot(data=df, x='Region', y='Ventas', palette='Set2', width=0.5, ax=axes[1,2])
sns.stripplot(data=df, x='Region', y='Ventas', color='black', alpha=0.3, size=3, ax=axes[1,2])
axes[1,2].set_title('boxplot + stripplot superpuesto')

for ax in axes.flatten():
    ax.set_xlabel('')
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout(); plt.show()


## 7. Pairplot — Explorar todas las relaciones de una vez

In [ ]:
# Genera automaticamente scatter de todas las combinaciones de variables.
# La diagonal muestra la distribucion de cada variable (KDE o histograma).
# hue= colorea por categoria: sirve para ver si los grupos se separan naturalmente.

cols = ['Ventas', 'Satisfaccion_Cliente', 'Costo_Operativo', 'Margen']
g = sns.pairplot(df[cols+['Region']], hue='Region', diag_kind='kde',
                 plot_kws={'alpha':0.5,'s':20}, palette='Set2')
g.figure.suptitle('Pairplot: todas las relaciones entre variables numericas',
                  y=1.02, fontsize=13, fontweight='bold')
plt.show()


## 8. Guardar un grafico como imagen

In [ ]:
# savefig() debe ir ANTES de plt.show() porque show() vacia la figura.
# dpi=150: suficiente para pantalla | dpi=300: calidad de impresion
# bbox_inches='tight': no corta titulos ni leyendas

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x='Region', y='Ventas', palette='Set2', ax=ax)
ax.set_title('Ventas por Region')
ax.grid(True, axis='y', linestyle='--', alpha=0.4)

fig.savefig('ventas_boxplot.png', dpi=150, bbox_inches='tight')
fig.savefig('ventas_boxplot.pdf', bbox_inches='tight')
plt.show()
print('Archivos guardados en la carpeta actual')


---
# Resumen: Clases 01 a 05

| Clase | Concepto clave | Por que importa |
|-------|---------------|------------------|
| 01 | Pipeline Data Science | Entender las etapas evita errores de orden |
| 02 | Python + NumPy vectorizado | Calculo rapido sin loops explicitos |
| 03 | Pandas DataFrame | Manipulacion de tablas con nombres de columnas |
| 04 | Estadistica descriptiva | Entender los datos antes de modelarlos |
| 04 | NaN: fillna, ffill, bfill | Datos sucios producen modelos sucios (GIGO) |
| 05 | Matplotlib + Seaborn | Visualizar antes de modelar |

```
Flujo tipico:
1. pd.read_csv()                           -> ingestar
2. df.describe() + isnull()                -> inspeccionar
3. mean/median/std/IQR/corr               -> estadistica descriptiva
4. str.strip() + fillna() + drop_dupl()    -> limpiar
5. groupby() + columnas calculadas         -> transformar
6. histplot / boxplot / heatmap / pairplot -> visualizar
```